# 00 — Execução Completa do Projeto

**Notebook orquestrador que executa todo o pipeline em um lugar:**
1. Baseline (Fase 1) — reproduz os 5 modelos clássicos
2. Algoritmo Genético (AG) — otimiza hiperparâmetros
3. Análise do AG — gera tabelas comparativas
4. LLM — gera explicações textuais com barra de progresso

Execute `Cell → Run All` para rodar tudo, ou célula por célula.

## Setup: Imports e Configuração

In [1]:
import subprocess
import json
import statistics
from pathlib import Path
import sys
from datetime import datetime
from tqdm import tqdm
import time

# Calcular caminhos corretamente (notebook está em fase2/notebooks/)
NOTEBOOK_DIR = Path.cwd()  # fase2/notebooks/
FASE2_DIR = NOTEBOOK_DIR.parent  # fase2/
PROJECT_ROOT = FASE2_DIR.parent  # raiz
FASE1_DIR = PROJECT_ROOT / 'fase1'
SAIDA_DIR = FASE2_DIR / 'arquivos_saida'
ANALITICOS_DIR = SAIDA_DIR / 'arquivos_analiticos'
RUNS_DIR = ANALITICOS_DIR / 'historico_execucoes_ag'
LOGS_DIR = SAIDA_DIR / 'logs'

# As etapas pesadas rodam por subprocess com cwd=FASE2_DIR, mas as células de
# análise importam src diretamente. O kernel sobe em notebooks/, então sem isto
# `import src` falha com ModuleNotFoundError.
if str(FASE2_DIR) not in sys.path:
    sys.path.insert(0, str(FASE2_DIR))

# As 12 execuções principais do §3.1 do relatório. São as pastas que este notebook
# grava, as mesmas que o notebook 01 e o notebook 02 leem: um caminho de saída só.
SEMENTES = (42, 7, 2024)
EXPERIMENTOS = ('E1_baseline', 'E2_exploracao', 'E3_aproveitamento', 'D_roleta')
EXECUCOES_AG = [f'{e}_seed{s}' for e in EXPERIMENTOS for s in SEMENTES]

print(f'✅ Ambiente pronto')
print(f'   Fase 1: {FASE1_DIR.exists()}')
print(f'   Fase 2: {FASE2_DIR.exists()}')
print(f'   {len(EXECUCOES_AG)} execuções de AG previstas')
print(f'\n⏰ {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


✅ Ambiente pronto
   Fase 1: True
   Fase 2: True
   12 execuções de AG previstas

⏰ 2026-10-17 22:32:57


## 1. Baseline (Fase 1)

**O quê:** Treina 5 modelos clássicos (Random Forest, Logistic Regression, etc.) no dataset original de SOP.  
**Por quê:** Estabelece a linha de base — a métrica que o AG vai tentar melhorar.  
**Em cima de:** Dataset PCOS de 541 pacientes (541 × 49 features).  
**Objetivo:** Verificar que os modelos reproduzem exatamente os mesmos scores da Fase 1 (F1, AUC, Recall).

In [2]:
print('🔄 Reproduzindo baseline (Fase 1)...\n')
result = subprocess.run([sys.executable, '-m', 'src.run_ml'], cwd=FASE1_DIR, capture_output=True, text=True)

if result.returncode == 0:
    print('✅ Baseline reproduzido\n')
    
    metrics_file = FASE1_DIR / 'reports' / 'metricas_modelos.json'
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    print('5 Modelos treinados:')
    for model_data in metrics[:5]:
        modelo = model_data.get('modelo', 'N/A')
        f1 = model_data.get('f1_score', 'N/A')
        recall = model_data.get('recall', 'N/A')
        if isinstance(f1, (int, float)) and isinstance(recall, (int, float)):
            print(f"  • {modelo:20s}: F1={f1:.4f}, Recall={recall:.4f}")
else:
    print(f'❌ Erro: {result.stderr}')

🔄 Reproduzindo baseline (Fase 1)...



✅ Baseline reproduzido

5 Modelos treinados:
  • Regressão Logística : F1=0.7949, Recall=0.8611
  • Árvore de Decisão   : F1=0.7887, Recall=0.7778
  • KNN (k=5)           : F1=0.7619, Recall=0.6667
  • Random Forest       : F1=0.8657, Recall=0.8056
  • SVM (RBF)           : F1=0.8219, Recall=0.8333


## 2. Algoritmo Genético (AG)

**O quê:** Roda 4 experimentos (E1, E2, E3, D) × 3 sementes cada = 12 execuções do AG.  
**Por quê:** Cada experimento testa uma configuração diferente (tamanho da população, taxa de mutação, operador de seleção).  
**Em cima de:** Hiperparâmetros codificados em cromossomos de 24 genes para cada um dos 5 modelos.  
**Objetivo:** Encontrar os melhores hiperparâmetros que maximizam F1+Recall+AUC em validação cruzada (k=5).
> **Esta célula regrava os artefatos oficiais, de propósito.** As 12 pastas
> `<experimento>_seed<semente>` em
> `arquivos_saida/arquivos_analiticos/historico_execucoes_ag/` são as que sustentam
> as tabelas do relatório, do notebook 01 e do notebook 02. Este notebook grava
> exatamente nelas: não faz sentido o notebook 00 escrever num lugar e o notebook 01
> em outro quando os dois rodam a mesma configuração. O AG é determinístico por
> semente, então reexecutar reproduz os mesmos números — só `tempo_total_s` e os
> campos de ambiente mudam, porque são medidos na hora.
>
> `--n-jobs=-1` reproduz o paralelismo das execuções que geraram os números do
> relatório. Com `n_jobs=1` o resultado por geração é idêntico, mas o tempo de
> parede triplica.
>
> Para rodar sem tocar nos artefatos oficiais, acrescente `--sufixo _teste`: a saída
> vai para `E1_baseline_seed42_teste` e irmãs. Isso cobre só a etapa do AG — a célula
> da etapa de LLM, mais abaixo, grava sempre nos nomes oficiais. Para isolá-la também,
> acrescente `'--sufixo', '_teste'` à chamada de `src.llm.rodar_avaliacao`.
>
> O console do AG é gravado em `arquivos_saida/logs/notebook00.out`, não num pipe.
> Com `stdout=PIPE` e um laço que só chama `poll()`, o buffer de pipe do sistema
> (64 KB no macOS) enche, o processo filho bloqueia dentro de `write()` e `poll()`
> nunca deixa de devolver `None`: a célula gira para sempre sem que o AG avance.
> Se a barra ficar parada, abra o log para ver em que geração a execução está. Se o
> processo travar mesmo, interromper a célula não encerra o AG: ele continua como
> processo filho, e é preciso encerrá-lo pelo PID ou reiniciar o kernel.


In [3]:
print('🧬 Rodando AG (4 experimentos × 3 sementes = 12 execuções)\n')
print('Configurações (as mesmas de src/experimentos/configuracoes.py):')
print('  E1: população=30, mutação=0.15, seleção=torneio k=2, gerações=15')
print('  E2: população=60, mutação=0.35, seleção=torneio k=2, gerações=15')
print('  E3: população=15, mutação=0.05, seleção=torneio k=4, gerações=15')
print('  D: população=30, mutação=0.15, seleção=ROLETA (ablação), gerações=15\n')
print('Referência: 446 s só de AG; 484 s com a avaliação no holdout, que esta célula faz.\n')
print('Saída nas 12 pastas oficiais — as mesmas que os notebooks 01 e 02 leem.\n')

LOG_NB = LOGS_DIR / 'notebook00.out'
LOG_NB.parent.mkdir(parents=True, exist_ok=True)

start_time = time.time()


def execucoes_gravadas():
    """Quantas das 12 execuções foram gravadas depois do início desta célula."""
    total = 0
    for nome in EXECUCOES_AG:
        resumo = RUNS_DIR / nome / 'resumo.json'
        if resumo.exists() and resumo.stat().st_mtime >= start_time:
            total += 1
    return total


with open(LOG_NB, 'w') as saida:
    process = subprocess.Popen(
        [sys.executable, '-m', 'src.experimentos.rodar_ag',
         '--experimento', 'todos', '--n-jobs=-1'],
        cwd=FASE2_DIR,
        stdout=saida,
        stderr=subprocess.STDOUT,
        text=True
    )

    # mininterval casa com o sleep: cada refresh da tqdm é salvo como uma saída no
    # .ipynb, e refrescar de segundo em segundo gera centenas de quadros inúteis.
    with tqdm(total=100, desc='AG', unit='%', ncols=70, leave=True,
              mininterval=15.0) as pbar:
        while process.poll() is None:
            feitas = execucoes_gravadas()
            # A barra conta execuções gravadas, não tempo decorrido: uma estimativa
            # de duração errada travaria a barra num valor fixo até o fim.
            pbar.n = int(feitas / len(EXECUCOES_AG) * 100)
            pbar.set_postfix_str(f'{feitas}/{len(EXECUCOES_AG)} execuções')
            pbar.refresh()
            time.sleep(15.0)
        pbar.n = 100
        pbar.refresh()

stderr = LOG_NB.read_text()[-3000:]  # cauda do log, usada só em caso de erro
print(f'Log completo da execucao: {LOG_NB.relative_to(FASE2_DIR)}')

if process.returncode == 0:
    print(f'\n✅ AG completo em {time.time() - start_time:.0f}s\n')
    print(f'📁 {execucoes_gravadas()}/12 execuções gravadas em '
          f'arquivos_saida/arquivos_analiticos/historico_execucoes_ag/')
    # Média ± desvio entre as 3 sementes: são estes os números do §3.2 do relatório.
    # O máximo de uma semente só não é comparável com a tabela de lá.
    print('\nFitness por experimento (média ± desvio entre as 3 sementes):')
    for exp_name in EXPERIMENTOS:
        fitness = []
        for semente in SEMENTES:
            resumo_file = RUNS_DIR / f'{exp_name}_seed{semente}' / 'resumo.json'
            if resumo_file.exists():
                with open(resumo_file) as f:
                    fitness.append(json.load(f)['melhor_fitness'])
        if fitness:
            media = statistics.mean(fitness)
            desvio = statistics.pstdev(fitness)
            print(f'  • {exp_name:18} {media:.4f} ± {desvio:.4f}'
                  f'   (melhor das {len(fitness)} sementes: {max(fitness):.4f})')
else:
    print(f'❌ Erro: {stderr}')


🧬 Rodando AG (4 experimentos × 3 sementes = 12 execuções)

Configurações (as mesmas de src/experimentos/configuracoes.py):
  E1: população=30, mutação=0.15, seleção=torneio k=2, gerações=15
  E2: população=60, mutação=0.35, seleção=torneio k=2, gerações=15
  E3: população=15, mutação=0.05, seleção=torneio k=4, gerações=15
  D: população=30, mutação=0.15, seleção=ROLETA (ablação), gerações=15

Referência: 446 s só de AG; 484 s com a avaliação no holdout, que esta célula faz.

Saída nas 12 pastas oficiais — as mesmas que os notebooks 01 e 02 leem.



AG:   0%|                                      | 0/100 [00:00<?, ?%/s]

AG:   0%|                      | 0/100 [00:00<?, ?%/s, 0/12 execuções]

AG:   0%|                      | 0/100 [00:00<?, ?%/s, 0/12 execuções]

AG:   0%|                      | 0/100 [00:15<?, ?%/s, 0/12 execuções]

AG:   0%|                      | 0/100 [00:15<?, ?%/s, 0/12 execuções]

... (59 outputs left out)

AG:  91%|██████████▉ | 91/100 [07:45<00:46,  5.12s/%, 11/12 execuções]

AG: 100%|███████████| 100/100 [08:00<00:00,  4.81s/%, 11/12 execuções]

AG: 100%|███████████| 100/100 [08:00<00:00,  4.81s/%, 11/12 execuções]

Log completo da execucao: arquivos_saida/logs/notebook00.out

✅ AG completo em 481s

📁 12/12 execuções gravadas em arquivos_saida/arquivos_analiticos/historico_execucoes_ag/

Fitness por experimento (média ± desvio entre as 3 sementes):
  • E1_baseline        0.8765 ± 0.0039   (melhor das 3 sementes: 0.8819)
  • E2_exploracao      0.8804 ± 0.0012   (melhor das 3 sementes: 0.8819)
  • E3_aproveitamento  0.8715 ± 0.0032   (melhor das 3 sementes: 0.8759)
  • D_roleta           0.8746 ± 0.0037   (melhor das 3 sementes: 0.8791)


## 3. Análise do AG

**O quê:** Lê os 12 arquivos de execução (history.jsonl) e gera tabelas comparativas.  
**Por quê:** Mostrar se o AG realmente melhorou versus o baseline e versus busca aleatória.  
**Em cima de:** Artefatos em `arquivos_saida/arquivos_analiticos/historico_execucoes_ag/` — configuração, histórico por geração, best individual.  
**Objetivo:** Comprovar se E1, E2, E3 bateram random search com menos treinos de modelo.
> **O que esta célula analisa.** `src/experimentos/analise.py` lê as execuções
> pelos nomes oficiais — `E1_baseline_seed42` e irmãs, as 12 que sustentam as
> tabelas do relatório e do notebook 02. São exatamente as pastas que a célula
> anterior acabou de gravar, então as tabelas abaixo descrevem a execução que
> você rodou agora.
>
> As funções de `analise.py` recebem sementes, não nomes de pasta. Se você usou
> `--sufixo` na célula 6, a análise continua lendo as pastas sem sufixo, ou seja,
> a execução oficial anterior, e não a sua.
>
> A comparação com a busca aleatória usa `C_busca_aleatoria_seed*`, que a célula 6
> não regrava: a busca aleatória é a Etapa C, rodada por
> `python -m src.experimentos.rodar_ag --etapa-c`.

In [4]:
print('📊 Analisando AG (baseline vs E1, E2, E3, D vs Random Search)\n')

result = subprocess.run([sys.executable, '-m', 'src.experimentos.analise'],
                       cwd=FASE2_DIR, capture_output=True, text=True)

if result.returncode == 0:
    lines = result.stdout.split('\n')
    for line in lines[:60]:
        print(line)
    
    if len(lines) > 60:
        print(f'\n... ({len(lines) - 60} linhas omitidas)\n')
        for line in lines[-5:]:
            print(line)
else:
    print(f'❌ Erro: {result.stderr}')

📊 Analisando AG (baseline vs E1, E2, E3, D vs Random Search)




Experimentos principais (média ± desvio sobre as sementes)
      Experimento  Fitness (média)  Fitness (desvio)  Ger. p/ 99% do máx  Gerações rodadas  Entropia final  Gap melhor-média  Tempo (s)  Cache hit
      E1_baseline         0.876478          0.003919            1.333333         12.000000        0.613712          0.054738  35.239729   0.137778
    E2_exploracao         0.880431          0.001154            0.666667         11.333333        1.298725          0.080102  82.763749   0.049015
E3_aproveitamento         0.871474          0.003170            1.000000         12.000000        0.235573          0.045545  19.408338   0.606667
         D_roleta         0.874556          0.003687            0.666667          6.666667        1.509270          0.086508  21.353968   0.154630

Original vs. otimizado por família (Etapa A)
             Modelo  F1 orig  F1 otim      Δ F1  Recall orig  Recall otim  Δ Recall  AUC orig  AUC otim     Δ AUC  Score orig  Score otim   Δ Score
      Rando

## 4. LLM — Escolha de Provedor

**O quê:** Gera explicações textuais para 15 pacientes usando 3 templates de prompt diferentes.  
**Por quê:** Avaliar se o modelo consegue justificar suas previsões de forma clara, completa e sem alucinar números.  
**Em cima de:** Probabilidades já previstas pelo melhor modelo do AG, contexto clínico (features relevantes).  
**Objetivo:** Gerar 45 saídas (15 pacientes × 3 templates) e verificar: alucinação, estrutura, termos proibidos.

---

**Escolha o provedor abaixo (mude na célula seguinte):**

In [5]:
# Ollama local, o provedor das execuções documentadas no §5 do relatório.
# Trocar para 'simulado' roda sem Ollama, com respostas de template fixo: serve
# para testar o encadeamento, não para reproduzir os números do relatório.
PROVEDOR = "ollama"
MODELO = "llama3.2:3b"
TEMPERATURA = 0.2
SEMENTE_LLM = 42
N_POR_ESTRATO = 5

In [6]:
print(f'🤖 Avaliação com LLM — provedor={PROVEDOR}, modelo={MODELO}\n')
print('15 pacientes do holdout (5 por estrato) × 3 templates = 45 saídas.')
print('Referência: 24 min no Ollama local, ~32 s por saída.\n')

LOG_LLM = LOGS_DIR / 'notebook00_llm.out'
LOG_LLM.parent.mkdir(parents=True, exist_ok=True)

inicio_llm = time.time()

with open(LOG_LLM, 'w') as saida:
    proc_llm = subprocess.Popen(
        [sys.executable, '-m', 'src.llm.rodar_avaliacao',
         '--provedor', PROVEDOR,
         '--modelo', MODELO,
         '--temperatura', str(TEMPERATURA),
         '--semente', str(SEMENTE_LLM),
         '--n-por-estrato', str(N_POR_ESTRATO)],
        # Sem --sufixo: grava em avaliacao_ollama_*, os nomes oficiais que o §5.4 e o
        # §5.5 citam e que o notebook 03 também usa. Acrescente '--sufixo', '_teste'
        # para isolar uma execução de prova.
        cwd=FASE2_DIR,
        stdout=saida,
        stderr=subprocess.STDOUT,
        text=True
    )

    with tqdm(total=45, desc='Saídas', unit='saída', ncols=70, leave=True,
              mininterval=15.0) as pbar:
        while proc_llm.poll() is None:
            # Cada saída gerada imprime uma linha '[nn/45] ...' no log.
            pbar.n = min(LOG_LLM.read_text().count('/45]'), 45)
            pbar.refresh()
            time.sleep(15.0)
        pbar.n = min(LOG_LLM.read_text().count('/45]'), 45)
        pbar.refresh()

print(f'\nLog completo: {LOG_LLM.relative_to(FASE2_DIR)}')

if proc_llm.returncode == 0:
    print(f'✅ Avaliação completa em {time.time() - inicio_llm:.0f}s')
else:
    print(f'❌ Erro (código {proc_llm.returncode}). Cauda do log:')
    print(LOG_LLM.read_text()[-3000:])
    raise RuntimeError('a avaliação com LLM falhou; veja o log acima')

🤖 Avaliação com LLM — provedor=ollama, modelo=llama3.2:3b

15 pacientes do holdout (5 por estrato) × 3 templates = 45 saídas.
Referência: 24 min no Ollama local, ~32 s por saída.



Saídas:   0%|                               | 0/45 [00:00<?, ?saída/s]

Saídas:   0%|                               | 0/45 [00:00<?, ?saída/s]

Saídas:   0%|                               | 0/45 [00:15<?, ?saída/s]

Saídas:   0%|                               | 0/45 [00:30<?, ?saída/s]

Saídas:   2%|▌                      | 1/45 [00:45<33:04, 45.09s/saída]

... (83 outputs left out)

Saídas:  98%|█████████████████████▌| 44/45 [21:46<00:29, 29.69s/saída]

Saídas: 100%|██████████████████████| 45/45 [22:01<00:00, 29.37s/saída]

Saídas: 100%|██████████████████████| 45/45 [22:01<00:00, 29.37s/saída]


Log completo: arquivos_saida/logs/notebook00_llm.out
✅ Avaliação completa em 1322s


## 5. Verificação das Saídas

**O quê:** Verifica cada uma das 45 saídas em 3 critérios automáticos.  
**Por quê:** Garantir que o LLM não está alucinando números, mantém estrutura exigida, respeita termos proibidos.  
**Em cima de:** Textos gerados (JSON), contexto original, verificador com 3 regras.  
**Objetivo:** Medir qualidade (0% alucinação = sucesso; 100% estrutura = sucesso).

In [7]:
# Reproduz a tabela do §5.4 do relatório a partir do JSON que acabou de ser gravado.
# As métricas vêm de resumir_metricas_automaticas, a mesma função que o notebook 03 usa.
from src.llm.avaliacao import resumir_metricas_automaticas, SaidaAvaliada

ARQUIVO_SAIDAS = ANALITICOS_DIR / 'avaliacao_llm' / f'avaliacao_{PROVEDOR}_saidas.json'
registros = json.loads(ARQUIVO_SAIDAS.read_text(encoding='utf-8'))
saidas = [SaidaAvaliada(**r) for r in registros]
resumo_llm = resumir_metricas_automaticas(saidas)

print(f'📊 Métricas automáticas — {len(saidas)} saídas em {len(resumo_llm)} templates')
print(f'   Fonte: {ARQUIVO_SAIDAS.relative_to(FASE2_DIR)}\n')
print(f'{"template":26} {"n":>3} {"alucin.":>8} {"aderên.":>8} {"proib.":>7} '
      f'{"chars":>7} {"latên.":>8}')
print('-' * 72)
for template, m in resumo_llm.items():
    print(f'{template:26} {m["n_amostras"]:>3} '
          f'{m["taxa_alucinacao_numerica_media"]:>8.3f} '
          f'{m["aderencia_estrutural_media"]:>8.3f} '
          f'{m["n_saidas_com_termo_proibido"]:>7} '
          f'{m["comprimento_medio_caracteres"]:>7.0f} '
          f'{m["latencia_media_s"]:>7.1f}s')
print('\nalucin. e aderên. são frações de 0 a 1 (média por saída), não porcentagens.')
print('proib. = nº de saídas com algum termo da lista proibida do §5.5.')

ARQUIVOS_LLM = sorted((ANALITICOS_DIR / 'avaliacao_llm').glob(f'avaliacao_{PROVEDOR}_*'))
print('\n📁 Artefatos:')
for arquivo in ARQUIVOS_LLM:
    print(f'   • {arquivo.name}  ({arquivo.stat().st_size:,} bytes)')
print('\nO CSV da rubrica não é regravado por esta execução: ele guarda as notas '
      'preenchidas à mão do §5.6.')

📊 Métricas automáticas — 45 saídas em 3 templates
   Fonte: arquivos_saida/arquivos_analiticos/avaliacao_llm/avaliacao_ollama_saidas.json

template                     n  alucin.  aderên.  proib.   chars   latên.
------------------------------------------------------------------------
v1_zero_shot                15    0.007    1.000       1    1769    30.4s
v2_few_shot                 15    0.000    1.000       0    1432    28.3s
v3_few_shot_estruturado     15    0.000    1.000       0    1333    28.8s

alucin. e aderên. são frações de 0 a 1 (média por saída), não porcentagens.
proib. = nº de saídas com algum termo da lista proibida do §5.5.



📁 Artefatos:
   • avaliacao_ollama_rubrica.csv  (2,556 bytes)
   • avaliacao_ollama_saidas.json  (195,129 bytes)
   • avaliacao_ollama_saidas.md  (83,066 bytes)

O CSV da rubrica não é regravado por esta execução: ele guarda as notas preenchidas à mão do §5.6.


## 6. Resumo Final

In [8]:
print('='*70)
print('✅ EXECUÇÃO COMPLETA')
print('='*70)
print(f'\n⏰ Fim: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

execucoes = sorted(d.name for d in RUNS_DIR.iterdir() if d.is_dir())
print('\n📦 Artefatos gerados:')
print(f'   1. Modelos ML (Fase 1): {FASE1_DIR / "reports"}/')
print(f'   2. {len(execucoes)} execuções de AG em '
      f'arquivos_saida/arquivos_analiticos/historico_execucoes_ag/')
print(f'      {len(EXECUCOES_AG)} delas são as oficiais do §3.1 regravadas por este notebook')
print(f'   3. {len(ARQUIVOS_LLM)} artefatos de LLM em '
      f'arquivos_saida/arquivos_analiticos/avaliacao_llm/avaliacao_{PROVEDOR}_*')
print(f'   4. Relatório técnico: documentacao/relatorio-tecnico.md')
print('\n📝 Próximos passos:')
print('   1. Revisar os notebooks 01, 02 e 03 para a análise detalhada')
print('   2. Rever a rubrica do §5.6 em avaliacao_{}_rubrica.csv (notas por template)'.format(PROVEDOR))
print('   3. Conferir as tabelas do relatório contra os números acima')

✅ EXECUÇÃO COMPLETA

⏰ Fim: 2026-10-17 23:03:02

📦 Artefatos gerados:
   1. Modelos ML (Fase 1): /Users/icaroamerico/Documents/pos-tech-fiap-ia-dev/fase1/reports/
   2. 21 execuções de AG em arquivos_saida/arquivos_analiticos/historico_execucoes_ag/
      12 delas são as oficiais do §3.1 regravadas por este notebook
   3. 3 artefatos de LLM em arquivos_saida/arquivos_analiticos/avaliacao_llm/avaliacao_ollama_*
   4. Relatório técnico: documentacao/relatorio-tecnico.md

📝 Próximos passos:
   1. Revisar os notebooks 01, 02 e 03 para a análise detalhada
   2. Rever a rubrica do §5.6 em avaliacao_ollama_rubrica.csv (notas por template)
   3. Conferir as tabelas do relatório contra os números acima
